# Notebook: `app.ipynb`
Auto-generated notebook with cell-by-cell breakdown.

## Setup & Imports
Let's start by bringing in all the necessary libraries and modules we'll need for this part of the project.

In [ ]:
import os
import io
import base64
import numpy as np
import tensorflow as tf
from PIL import Image
from flask import Flask, render_template, request, jsonify

# Add src to path if running from root
import sys


## Path Setup
Adjusting the python path so we can import local modules.

In [ ]:
sys.path.append(os.path.join(os.path.dirname(__file__), 'src'))


## Setup & Imports
Let's start by bringing in all the necessary libraries and modules we'll need for this part of the project.

In [ ]:
from grad_cam import GradCAM


## Configuration & Constants
Here we define some global configuration settings and constants used throughout the code.

In [ ]:
app = Flask(__name__)

# Constants
MODEL_PATH = 'models/dummy_model.h5'
CLASSES = ['Actinic keratoses (akiec)', 'Basal cell carcinoma (bcc)', 
           'Benign keratosis-like lesions (bkl)', 'Dermatofibroma (df)', 
           'Melanoma (mel)', 'Melanocytic nevi (nv)', 'Vascular lesions (vasc)']
IMG_SIZE = (224, 224)

# Load model globally
model = None


## Model Loading utility
This helper function loads our pre-trained skin disease model into memory. We cache it globally so we don't have to reload it for every single prediction request.

In [ ]:
def load_cached_model():
    global model
    if model is None:
        if os.path.exists(MODEL_PATH):
            print(f"Loading model from {MODEL_PATH}")
            model = tf.keras.models.load_model(MODEL_PATH)
        else:
            print("Model not found. Please run src/generate_dummy_model.py or train a model.")
    return model


## Image Preprocessing
Before feeding an image to our model, we need to preprocess it (resize to 224x224, convert to array, expand dimensions) just like we did during training. We also prepare a resized version of the original image for the Grad-CAM overlay.

In [ ]:
def prepare_image(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img_orig = np.array(img)
    
    img = img.resize(IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    
    # Preprocessing expected by EfficientNetV2 is handled internally, 
    # but we need batch dimension
    img_array = np.expand_dims(img_array, axis=0)
    
    # Original image resized for GradCAM overlay
    img_orig_resized = cv2.resize(img_orig, IMG_SIZE) if 'cv2' in sys.modules else np.array(img.resize(IMG_SIZE))
    
    return img_array, img_orig_resized


## Base64 Encoding
We'll use this small utility function to convert our Grad-CAM heatmap images into base64 strings so they can be sent directly via the JSON API to the frontend.

In [ ]:
def encode_image_to_base64(img_array):
    img = Image.fromarray(img_array)
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return img_str


## Main Web Interface
This route serves the main front-end application (HTML page).

In [ ]:
def index():
    return render_template('index.html')


## Prediction Endpoint
This is the core API endpoint. It receives an uploaded image, runs it through our EfficientNetV2 model, generates a Grad-CAM heatmap to explain the prediction, and returns everything as JSON.

In [ ]:
def predict():
    try:
        if 'image' not in request.files:
            return jsonify({'error': 'No image uploaded'})
            
        file = request.files['image']
        if file.filename == '':
            return jsonify({'error': 'No selected file'})
            
        img_bytes = file.read()
        
        import cv2 # ensure cv2 is available here
        model = load_cached_model()
        if model is None:
            return jsonify({'error': 'Model not loaded. Server error.'})
            
        # Prepare image
        processed_img, orig_img = prepare_image(img_bytes)
        
        # Predict
        preds = model.predict(processed_img)[0]
        class_idx = np.argmax(preds)
        confidence = float(preds[class_idx])
        pred_class = CLASSES[class_idx]
        
        # Generate Grad-CAM
        try:
            cam = GradCAM(model, class_idx)
            heatmap = cam.compute_heatmap(processed_img)
            overlay = cam.overlay_heatmap(heatmap, orig_img)
            
            # Encode overlay to base64
            heatmap_b64 = encode_image_to_base64(overlay)
        except Exception as e:
            print(f"GradCAM error: {e}")
            heatmap_b64 = None
            
        return jsonify({
            'success': True,
            'prediction': pred_class,
            'confidence': f"{confidence:.2%}",
            'heatmap': heatmap_b64
        })
        
    except Exception as e:
        return jsonify({'error': str(e)})


## Execution
Finally, this block runs the code if the script is executed directly.

In [ ]:
if __name__ == '__main__':
    # Initialize model on startup
    load_cached_model()
    app.run(debug=True, port=5000)
